In [3]:
import csv
import ast
import matplotlib.pyplot as plt
from collections import Counter
import nltk
from itertools import islice
nltk.download("punkt")
nltk.download("stopwords")
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

import numpy as np
import torch
from torch import nn
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import f1_score, precision_recall_fscore_support

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
!wget https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv
!wget https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/refs/heads/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv
!wget https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/refs/heads/master/semeval-2022/practice%20splits/train_semeval_parids-labels.csv

--2026-03-04 17:05:10--  https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3122842 (3.0M) [text/plain]
Saving to: ‘dontpatronizeme_pcl.tsv.1’

dontpatronizeme_pcl 100%[===================>]   2.98M  --.-KB/s    in 0.07s   

2026-03-04 17:05:10 (45.5 MB/s) - ‘dontpatronizeme_pcl.tsv.1’ saved [3122842/3122842]

--2026-03-04 17:05:10--  https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/refs/heads/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubus

In [6]:
paragraphs = {}

with open("dontpatronizeme_pcl.tsv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for entry in islice(reader, 4, None):
        paragraphs[entry[0]] = entry[4]

In [7]:
train_data = []
train_labels = []
dev_data = []
dev_labels = []

def parse_labels(filename):
    data = []
    labels = []
    with open(filename, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in islice(reader, 1, None):
            data.append(paragraphs[row[0]])
            labels.append(any(ast.literal_eval(row[1])))
    return data, labels

train_data, train_labels = parse_labels("train_semeval_parids-labels.csv")
dev_data, dev_labels = parse_labels("dev_semeval_parids-labels.csv")

In [8]:
train_texts_new, val_texts_new, train_labels_new, val_labels_new = train_test_split(
    train_data,
    train_labels,
    test_size=0.2,
    stratify=train_labels,   # keeps 90/10 ratio
    random_state=42
)

print("Train split:", Counter(train_labels_new))
print("Val split:", Counter(val_labels_new))

Train split: Counter({False: 6065, True: 635})
Val split: Counter({False: 1516, True: 159})


In [9]:
train_y = [1 if y else 0 for y in train_labels_new]
val_y   = [1 if y else 0 for y in val_labels_new]
dev_y = [1 if y else 0 for y in dev_labels]

In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_batch(texts, max_len=256):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_len
    )

train_enc = tokenize_batch(train_texts_new, max_len=128)
val_enc   = tokenize_batch(val_texts_new,   max_len=128)
dev_enc = tokenize_batch(dev_data)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.enc = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx]).long()
        return item

train_ds = TextDataset(train_enc, train_y)
val_ds   = TextDataset(val_enc, val_y)
dev_ds = TextDataset(dev_enc, dev_y)

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, labels=[0,1], average=None)
    return {
        "f1_pos": f1[1],
        "precision_pos": p[1],
        "recall_pos": r[1],
    }

In [14]:
class FocalLoss(nn.Module):
    """
    Multiclass focal loss (works for binary with num_labels=2).
    alpha: torch.tensor([w0, w1]) optional class weights (like CrossEntropyLoss weight)
    gamma: focusing parameter (2 is a common default)
    """
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        # logits: [B, C], targets: [B]
        ce = nn.CrossEntropyLoss(weight=self.alpha, reduction="none")
        ce_loss = ce(logits, targets)                 # [B]
        pt = torch.exp(-ce_loss)                      # [B]
        focal = ((1 - pt) ** self.gamma) * ce_loss    # [B]
        return focal.mean()

In [15]:
class FocalTrainer(Trainer):
    def __init__(self, focal_alpha=None, focal_gamma=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.focal_loss = None  # created lazily once we know device

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits

        # Create focal loss on correct device once
        if self.focal_loss is None:
            alpha = self.focal_alpha
            if alpha is not None:
                alpha = alpha.to(logits.device)
            self.focal_loss = FocalLoss(alpha=alpha, gamma=self.focal_gamma)

        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [13]:
training_args = TrainingArguments(
    output_dir="baseline_roberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to=[],
)

In [16]:
model_f = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
focal_alpha = torch.tensor([1.0, 5.0], dtype=torch.float) 

trainer_f = FocalTrainer(
    model=model_f,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,                 # internal val split (for early stopping)
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    focal_alpha=focal_alpha,             # or None
    focal_gamma=2.0,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer_f.train()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Pos,Precision Pos,Recall Pos
1,0.318158,0.296378,0.393352,0.252220,0.893082
2,0.217695,0.310438,0.575758,0.481013,0.716981
3,0.148858,0.452978,0.554348,0.488038,0.641509
4,0.089734,0.557337,0.568562,0.607143,0.534591
5,0.042486,0.630872,0.562092,0.585034,0.540881


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1050, training_loss=0.17426012754440307, metrics={'train_runtime': 514.6209, 'train_samples_per_second': 130.193, 'train_steps_per_second': 4.081, 'total_flos': 2203555088640000.0, 'train_loss': 0.17426012754440307, 'epoch': 5.0})

In [17]:
from sklearn.metrics import f1_score, precision_recall_fscore_support

# 1) threshold tune on internal val
pred_val = trainer_f.predict(val_ds)
logits_val = pred_val.predictions
labels_val = pred_val.label_ids

probs_val = torch.softmax(torch.tensor(logits_val), dim=1).numpy()
p_pos_val = probs_val[:, 1]

best_t, best_f1 = 0.5, -1.0
for t in np.linspace(0.01, 0.99, 99):
    yhat = (p_pos_val >= t).astype(int)
    f1 = f1_score(labels_val, yhat, pos_label=1)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print("Best val threshold:", best_t)
print("Best val F1:", best_f1)

# 2) evaluate on dev (your test)
pred_dev = trainer_f.predict(dev_ds)
logits_dev = pred_dev.predictions
labels_dev = pred_dev.label_ids

probs_dev = torch.softmax(torch.tensor(logits_dev), dim=1).numpy()
p_pos_dev = probs_dev[:, 1]
yhat_dev = (p_pos_dev >= best_t).astype(int)

p, r, f1, _ = precision_recall_fscore_support(labels_dev, yhat_dev, labels=[0,1], average=None)
print("Dev precision_pos:", p[1])
print("Dev recall_pos:", r[1])
print("Dev f1_pos:", f1[1])

Best val threshold: 0.5800000000000001
Best val F1: 0.5826330532212886


Dev precision_pos: 0.5122950819672131
Dev recall_pos: 0.628140703517588
Dev f1_pos: 0.5643340857787811


In [18]:
pred_val = trainer_f.predict(val_ds)
logits = pred_val.predictions
y_true = pred_val.label_ids

y_pred = np.argmax(logits, axis=1)

# False positives: true 0 but predicted 1
fp_idx = np.where((y_true == 0) & (y_pred == 1))[0]
print("False positives found:", len(fp_idx))

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
p_pos = probs[:, 1]

# Hard negatives = true 0 with high P(PCL)
hn_idx = np.where((y_true == 0) & (p_pos >= 0.6))[0]   # threshold you can tune (0.6–0.8)
print("Hard negatives (p_pos>=0.6):", len(hn_idx))

False positives found: 123
Hard negatives (p_pos>=0.6): 85


In [19]:
K = 3  # oversampling factor

# Choose indices: hn_idx or fp_idx
selected_idx = hn_idx

hard_texts = [val_texts_new[i] for i in selected_idx]
hard_labels = [0 for _ in selected_idx]  # they are true negatives

# Augmented training data
train_texts_aug = train_texts_new + hard_texts * K
train_y_aug     = train_y          + hard_labels * K

print("Original train size:", len(train_texts_new))
print("Augmented train size:", len(train_texts_aug))

Original train size: 6700
Augmented train size: 6955


In [22]:
train_enc_aug = tokenize_batch(train_texts_aug, max_len=256)
train_ds2 = TextDataset(train_enc, train_y_aug)

In [ ]:
model2 = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

trainer2 = FocalTrainer(
    model=model2,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,                 # internal val split (for early stopping)
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    focal_alpha=focal_alpha,             # or None
    focal_gamma=2.0,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer2.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Pos,Precision Pos,Recall Pos
1,0.324000,0.303705,0.385576,0.247331,0.874214
2,0.211211,0.310854,0.521951,0.426295,0.672956
3,0.172419,0.375805,0.533333,0.420290,0.729560
4,0.129324,0.558229,0.550633,0.554140,0.547170
5,0.036406,0.765546,0.511475,0.534247,0.490566


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]